In [4]:
"""
fill_blinded_review_zs.py
─────────────────────────────────────────────────────────────────────────────
Runs CONCH zero-shot, H-optimus-1 ANN, and CONCH few-shot (logistic
regression) inference on every patch listed in the "Blinded Review" sheet
of the aggregated pathologist-review workbook, then writes:

  • ZS Prediction  + ZS Confidence
  • ANN Prediction + ANN Confidence
  • FS Prediction  + FS Confidence
  • Patch thumbnail (Patch Image column)

back into that sheet.  New columns are appended to the right of whatever
columns already exist, so existing pathologist labels are never touched.

HOW IT FINDS THE PATCHES
  The blinded filenames are anonymised.  The script searches SEARCH_ROOTS
  recursively for files whose name exactly matches the blinded filename.
  Optionally supply a mapping_csv (columns: blinded_name, original_name) to
  resolve names that differ from the originals.

CONFIGURE THE BLOCK BELOW BEFORE RUNNING
─────────────────────────────────────────────────────────────────────────────
"""
import os
from dotenv import load_dotenv
load_dotenv()
import io, os, sys, json
from pathlib import Path

import numpy as np
import torch
import timm
import joblib
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login
from torchvision import transforms
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ══════════════════════════════════════════════════════════════════════════════
#  ▶▶  CONFIGURE EVERYTHING HERE  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

CFG = dict(

    # ── Input workbook ────────────────────────────────────────────────────────
    input_xlsx    = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\aggregated_both_Pathologist_Reviews.xlsx",
    blinded_sheet = "Blinded Review",

    # ── Column NAMES that already exist (used to find them dynamically) ───────
    col_name_patch_image = "Patch Image",   # thumbnail column
    col_name_filename    = "Filename",      # blinded patch name (read-only)
 
    header_row = 1,   # row with headers; data starts at header_row + 1
 
    # ── Names for the SIX inference columns to write ──────────────────────────
    # The script finds them by name in the header row.
    # If absent, it appends them at the end.  Existing pathologist columns
    # are NEVER overwritten.
    col_name_zs_pred   = "ZS Prediction",
    col_name_zs_conf   = "ZS Confidence",
    col_name_ann_pred  = "ANN Prediction",
    col_name_ann_conf  = "ANN Confidence",
    col_name_fs_pred   = "FS Prediction",
    col_name_fs_conf   = "FS Confidence",

    # ── Folders to search recursively for patch image files ──────────────────
    search_roots = [
        r"D:\Aamir Gulzar\dataset\CRC100K\CRC-VAL-HE-7K",
        r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM",
    ],

    # ── (Optional) blinded→original name mapping CSV ─────────────────────────
    mapping_csv = None,   # e.g. r"blinded_to_original_mapping.csv"

    # ── CONCH (zero-shot + few-shot features) ─────────────────────────────────
    conch_checkpoint = r"checkpoints\conch\pytorch_model.bin",
    zs_weights_path = ("D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\classifier_weights\Conch_zeroshot_weights.pt"),
    prompt_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\config_files\Custom_prompts_all_per_class.json",

    # ── Logistic Regression (few-shot linear probe, trained on CONCH features)
    lr_model_path = (r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
                     r"\slide_aggregation\caption_generation\Results"
                     r"\Few_Shot_Evaluation\180_trainset\few_shot_logistic_regression.pkl"),

    # ── H-optimus-1 (ANN backbone) ────────────────────────────────────────────
    hoptimus_model_id = "hf-hub:bioptimus/H-optimus-1",
    ann_model_path    = (r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
                         r"\TissueClassifier_CRC100K\models\ann_crc100k_Hoptimus.pth"),
    hf_token = os.environ.get("HF_TOKEN", ""), # paste if needed else put it as ""

    # ── ANN class order — must match what the ANN was trained on ─────────────
    ann_class_order = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"],

    # ── Thumbnail size embedded in Excel (px) ────────────────────────────────
    thumb_size = (125, 125),

    # ── Output workbook ───────────────────────────────────────────────────────
    output_xlsx = r"aggregated_both_Pathologist_Reviews_filled.xlsx",
)

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  COLUMN RESOLVER  (the core fix)
# ══════════════════════════════════════════════════════════════════════════════

def resolve_columns(ws, header_row, required_names: list[str]) -> dict[str, int]:
    """
    Returns {col_name: col_index_1based} for every name in required_names.

    For names already present in the header row → uses their existing column.
    For names NOT found → appends them one-by-one to the right of the last
    used column, so they never overwrite existing (pathologist) data.
    """
    # Build {normalised_label: col_idx} from the actual header row
    existing: dict[str, int] = {}
    last_used_col = 0
    for cell in ws[header_row]:
        if cell.value is not None:
            existing[str(cell.value).strip()] = cell.column
            last_used_col = max(last_used_col, cell.column)

    result: dict[str, int] = {}
    next_col = last_used_col + 1

    for name in required_names:
        if name in existing:
            result[name] = existing[name]
            print(f"  ✔ found existing column '{name}' at col {existing[name]}")
        else:
            result[name] = next_col
            print(f"  ✚ appending new column '{name}' at col {next_col}")
            next_col += 1

    return result

# ══════════════════════════════════════════════════════════════════════════════
#  MODEL HELPERS  (unchanged from original)
# ══════════════════════════════════════════════════════════════════════════════

def load_conch_model(checkpoint_path, device):
    from conch.open_clip_custom import create_model_from_pretrained
    print(f"  Loading CONCH from {checkpoint_path} …")
    model, preprocess = create_model_from_pretrained(
        model_cfg        = "conch_ViT-B-16",
        checkpoint_path  = checkpoint_path,
        device           = device,
        force_image_size = 224,
    )
    model.eval()
    return model, preprocess


def encode_conch(model, preprocess, image_path, device):
    img    = Image.open(image_path).convert("RGB")
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.inference_mode():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().float().squeeze(0)


def zs_predict(feat_cpu, zs_weights_cpu, zs_class_order):
    logits = 100.0 * feat_cpu.unsqueeze(0) @ zs_weights_cpu
    probs  = torch.softmax(logits, dim=-1)
    idx    = int(probs.argmax(dim=-1).item())
    return zs_class_order[idx], float(probs[0, idx].item())


def fs_predict(feat_cpu, lr_clf):
    x    = feat_cpu.unsqueeze(0).numpy()
    pred = lr_clf.predict(x)[0]
    conf = float(lr_clf.predict_proba(x).max())
    return pred, conf


def build_hoptimus_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.707223, 0.578729, 0.703617),
            std =(0.211883, 0.230117, 0.177517),
        ),
    ])


def load_hoptimus_model(model_id, hf_token, device):
    if hf_token:
        login(token=hf_token)
    print(f"  Loading H-optimus-1 ({model_id}) …")
    model = timm.create_model(
        model_id,
        pretrained       = True,
        init_values      = 1e-5,
        dynamic_img_size = False,
    ).to(device)
    model.eval()
    return model


def encode_hoptimus(model, transform, image_path, device):
    img    = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.inference_mode():
        emb = model(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().float().squeeze(0)


def ann_predict(feat_cpu, ann_model, ann_class_order, device):
    hf_dev  = feat_cpu.unsqueeze(0).to(device)
    with torch.inference_mode():
        out = ann_model(hf_dev).cpu()
    prob = torch.softmax(out, dim=-1)
    idx  = int(out.argmax(dim=-1).item())
    return ann_class_order[idx], float(prob[0, idx].item())

# ══════════════════════════════════════════════════════════════════════════════
#  FILE INDEX HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def build_filename_index(search_roots):
    index = {}
    exts  = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            print(f"  [WARN] search root not found: {root}")
            continue
        for p in root.rglob("*"):
            if p.suffix.lower() in exts:
                if p.name in index:
                    print(f"  [WARN] duplicate '{p.name}': overwriting {index[p.name]}")
                index[p.name] = str(p)
    print(f"  Indexed {len(index)} image files.")
    return index


def load_mapping_csv(csv_path):
    import csv
    mapping = {}
    with open(csv_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            mapping[row["blinded_name"].strip()] = row["original_name"].strip()
    print(f"  Mapping CSV: {len(mapping)} entries.")
    return mapping


def make_thumbnail_bytes(image_path, thumb_size):
    try:
        with Image.open(image_path) as im:
            im = im.convert("RGB")
            im.thumbnail(thumb_size, Image.LANCZOS)
            buf = io.BytesIO()
            im.save(buf, format="PNG")
            buf.seek(0)
            return buf
    except Exception as e:
        print(f"  [WARN] thumbnail failed for {image_path}: {e}")
        return None

# ══════════════════════════════════════════════════════════════════════════════
#  EXCEL HELPERS
# ══════════════════════════════════════════════════════════════════════════════

_SIDE   = Side(style="thin", color="BBBBBB")
_BORDER = Border(left=_SIDE, right=_SIDE, top=_SIDE, bottom=_SIDE)
_ALIGN  = Alignment(horizontal="center", vertical="center", wrap_text=True)
_FONT   = Font(name="Arial", size=9)
_FONT_H = Font(name="Arial", size=9, bold=True, color="FFFFFF")
_FILL_H = PatternFill("solid", fgColor="1F3864")


def _write(ws, row, col, value):
    c = ws.cell(row=row, column=col, value=value)
    c.font = _FONT; c.alignment = _ALIGN; c.border = _BORDER


def _write_header(ws, col, label, width=14):
    c = ws.cell(row=1, column=col)
    c.value     = label
    c.font      = _FONT_H
    c.fill      = _FILL_H
    c.alignment = _ALIGN
    c.border    = _BORDER
    ws.column_dimensions[get_column_letter(col)].width = width

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 65)
    print("  Blinded Review — Full Inference Fill Script  (fixed)")
    print("  (ZS + ANN + Few-Shot  |  dynamic column resolution)")
    print("=" * 65)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device : {device}\n")

    # ── 1. Open workbook ──────────────────────────────────────────────────────
    print(f"[1/7] Opening workbook …")
    wb = load_workbook(CFG["input_xlsx"])
    if CFG["blinded_sheet"] not in wb.sheetnames:
        sys.exit(f"  ERROR: sheet '{CFG['blinded_sheet']}' not found. "
                 f"Available: {wb.sheetnames}")
    ws = wb[CFG["blinded_sheet"]]
    header_row = CFG["header_row"]

    # ── Resolve all column positions dynamically ──────────────────────────────
    print(f"\n  Resolving column positions from header row {header_row} …")

    INFERENCE_COL_NAMES = [
        CFG["col_name_zs_pred"],
        CFG["col_name_zs_conf"],
        CFG["col_name_ann_pred"],
        CFG["col_name_ann_conf"],
        CFG["col_name_fs_pred"],
        CFG["col_name_fs_conf"],
    ]
    # Also find the read-only columns
    READ_ONLY_COL_NAMES = [
        CFG["col_name_patch_image"],
        CFG["col_name_filename"],
    ]

    all_cols = resolve_columns(ws, header_row,
                               READ_ONLY_COL_NAMES + INFERENCE_COL_NAMES)

    col_img  = all_cols[CFG["col_name_patch_image"]]
    col_fn   = all_cols[CFG["col_name_filename"]]
    col_zsp  = all_cols[CFG["col_name_zs_pred"]]
    col_zsc  = all_cols[CFG["col_name_zs_conf"]]
    col_annp = all_cols[CFG["col_name_ann_pred"]]
    col_annc = all_cols[CFG["col_name_ann_conf"]]
    col_fsp  = all_cols[CFG["col_name_fs_pred"]]
    col_fsc  = all_cols[CFG["col_name_fs_conf"]]

    # Write / refresh headers for the six inference columns
    _write_header(ws, col_zsp,  CFG["col_name_zs_pred"],  16)
    _write_header(ws, col_zsc,  CFG["col_name_zs_conf"],  14)
    _write_header(ws, col_annp, CFG["col_name_ann_pred"],  16)
    _write_header(ws, col_annc, CFG["col_name_ann_conf"],  14)
    _write_header(ws, col_fsp,  CFG["col_name_fs_pred"],  16)
    _write_header(ws, col_fsc,  CFG["col_name_fs_conf"],  14)

    print(f"\n  Final column mapping:")
    print(f"    Patch Image   → col {col_img}")
    print(f"    Filename      → col {col_fn}")
    print(f"    ZS Prediction → col {col_zsp}")
    print(f"    ZS Confidence → col {col_zsc}")
    print(f"    ANN Prediction→ col {col_annp}")
    print(f"    ANN Confidence→ col {col_annc}")
    print(f"    FS Prediction → col {col_fsp}")
    print(f"    FS Confidence → col {col_fsc}")

    # ── Read blinded rows ─────────────────────────────────────────────────────
    patch_rows = []
    for row in ws.iter_rows(min_row=header_row + 1, values_only=False):
        fn_cell = row[col_fn - 1]
        fn_val  = str(fn_cell.value).strip() if fn_cell.value else ""
        if not fn_val or fn_val.lower() in ("none", "nan", ""):
            continue
        patch_rows.append((fn_cell.row, fn_val))

    print(f"\n  Blinded patches found : {len(patch_rows)}")
    if not patch_rows:
        sys.exit("  Nothing to do.")

    # ── 2. Build file index ───────────────────────────────────────────────────
    print(f"\n[2/7] Indexing image files …")
    file_index = build_filename_index(CFG["search_roots"])
    mapping    = load_mapping_csv(CFG["mapping_csv"]) if CFG.get("mapping_csv") else {}

    resolved, unresolved = {}, []
    for _, fn in patch_rows:
        if fn in file_index:
            resolved[fn] = file_index[fn]
        elif fn in mapping and mapping[fn] in file_index:
            resolved[fn] = file_index[mapping[fn]]
        else:
            unresolved.append(fn)

    print(f"  Resolved   : {len(resolved)} / {len(patch_rows)}")
    if unresolved:
        print(f"  Unresolved : {len(unresolved)}")
        for fn in unresolved:
            print(f"    ✗ {fn}")

    if not resolved:
        sys.exit("  ERROR: no patches could be located. Aborting.")

    # ── 3. Load ZS class order ────────────────────────────────────────────────
    print(f"\n[3/7] Loading ZS class list …")
    with open(CFG["prompt_file"]) as fh:
        prompt_data = json.load(fh)["0"]
    ZS_CLASS_ORDER  = list(prompt_data["classnames"].keys())
    ANN_CLASS_ORDER = CFG["ann_class_order"]
    print(f"  ZS  classes ({len(ZS_CLASS_ORDER)}): {ZS_CLASS_ORDER}")
    print(f"  ANN classes ({len(ANN_CLASS_ORDER)}): {ANN_CLASS_ORDER}")

    # ── 4. Load CONCH + ZS weights ────────────────────────────────────────────
    print(f"\n[4/7] Loading CONCH model & ZS weights …")
    conch_model, preprocess = load_conch_model(CFG["conch_checkpoint"], device)

    zs_weights = torch.load(CFG["zs_weights_path"], map_location="cpu",
                            weights_only=False).float()
    if zs_weights.shape[0] == len(ZS_CLASS_ORDER):
        zs_weights = zs_weights.T
    assert zs_weights.shape[1] == len(ZS_CLASS_ORDER), (
        f"ZS weight cols ({zs_weights.shape[1]}) != ZS class count "
        f"({len(ZS_CLASS_ORDER)}). Check zs_weights_path / prompt_file."
    )
    print(f"  ZS weight matrix: {tuple(zs_weights.shape)}")

    # ── 5. Load few-shot logistic regression ──────────────────────────────────
    print(f"\n[5/7] Loading few-shot logistic regression …")
    lr_clf = joblib.load(CFG["lr_model_path"])
    print(f"  LR model loaded.")

    # ── 6. Load H-optimus-1 + ANN ────────────────────────────────────────────
    print(f"\n[6/7] Loading H-optimus-1 & ANN …")
    hop_transform = build_hoptimus_transform()
    hop_model     = load_hoptimus_model(CFG["hoptimus_model_id"],
                                        CFG.get("hf_token", ""), device)

    ann_model = torch.load(CFG["ann_model_path"], map_location=device,
                           weights_only=False)
    ann_model.to(device)
    ann_model.eval()
    print(f"  ANN model loaded.")

    # ── 7. Run inference & write back ─────────────────────────────────────────
    print(f"\n[7/7] Running inference on {len(resolved)} patches …")

    THUMB = CFG["thumb_size"]
    ROW_H = THUMB[1] + 10
    filled = skipped = 0

    for excel_row, fn in tqdm(patch_rows, desc="Patches"):
        if fn not in resolved:
            skipped += 1
            continue

        full_path = resolved[fn]

        try:
            # CONCH embedding shared by ZS and FS
            conch_feat = encode_conch(conch_model, preprocess, full_path, device)
            zs_pred,  zs_conf  = zs_predict(conch_feat, zs_weights, ZS_CLASS_ORDER)
            fs_pred,  fs_conf  = fs_predict(conch_feat, lr_clf)

            # H-optimus-1 → ANN
            hop_feat           = encode_hoptimus(hop_model, hop_transform, full_path, device)
            ann_pred, ann_conf = ann_predict(hop_feat, ann_model, ANN_CLASS_ORDER, device)

        except Exception as e:
            print(f"  [WARN] inference failed for {fn}: {e}")
            skipped += 1
            continue

        # Write all six inference values to the correct dynamically-resolved columns
        _write(ws, excel_row, col_zsp,  zs_pred)
        _write(ws, excel_row, col_zsc,  f"{zs_conf:.1%}")
        _write(ws, excel_row, col_annp, ann_pred)
        _write(ws, excel_row, col_annc, f"{ann_conf:.1%}")
        _write(ws, excel_row, col_fsp,  fs_pred)
        _write(ws, excel_row, col_fsc,  f"{fs_conf:.1%}")

        # Embed patch thumbnail
        ws.row_dimensions[excel_row].height = ROW_H
        thumb_buf = make_thumbnail_bytes(full_path, THUMB)
        if thumb_buf:
            img_col_idx = col_img - 1
            ws._images = [im for im in ws._images
                          if not (hasattr(im.anchor, "_from")
                                  and im.anchor._from.row == excel_row - 1
                                  and im.anchor._from.col == img_col_idx)]
            xl_img    = XLImage(thumb_buf)
            cell_addr = f"{get_column_letter(col_img)}{excel_row}"
            ws.add_image(xl_img, cell_addr)
            ws.cell(row=excel_row, column=col_img).value = None

        filled += 1

    # ── Save ──────────────────────────────────────────────────────────────────
    out = Path(CFG["output_xlsx"])
    out.parent.mkdir(parents=True, exist_ok=True)
    wb.save(str(out))

    print(f"\n{'=' * 65}")
    print(f"  ✅  Done!  Output → {out}")
    print(f"      Filled : {filled}  |  Skipped : {skipped}")
    print(f"{'=' * 65}")


if __name__ == "__main__":
    main()

  Blinded Review — Full Inference Fill Script  (fixed)
  (ZS + ANN + Few-Shot  |  dynamic column resolution)
  Device : cuda

[1/7] Opening workbook …

  Resolving column positions from header row 1 …
  ✔ found existing column 'Patch Image' at col 1
  ✔ found existing column 'Filename' at col 2
  ✔ found existing column 'ZS Prediction' at col 3
  ✔ found existing column 'ZS Confidence' at col 4
  ✚ appending new column 'ANN Prediction' at col 7
  ✚ appending new column 'ANN Confidence' at col 8
  ✚ appending new column 'FS Prediction' at col 9
  ✚ appending new column 'FS Confidence' at col 10

  Final column mapping:
    Patch Image   → col 1
    Filename      → col 2
    ZS Prediction → col 3
    ZS Confidence → col 4
    ANN Prediction→ col 7
    ANN Confidence→ col 8
    FS Prediction → col 9
    FS Confidence → col 10

  Blinded patches found : 60

[2/7] Indexing image files …
  Indexed 107180 image files.
  Resolved   : 60 / 60

[3/7] Loading ZS class list …
  ZS  classes (14): [

Patches: 100%|██████████| 60/60 [00:02<00:00, 27.17it/s]



  ✅  Done!  Output → aggregated_both_Pathologist_Reviews_filled.xlsx
      Filled : 60  |  Skipped : 0
